# 01 — Build the ZCTA / data-center modeling table

Goal: one row per ZCTA nationally, with Merascope's 23 indicators as features and
"does a data center sit here" as the label.

**Features** come from the live product's own output,
`data/{STATE}/zcta/grid_scores.geojson`, pulled over HTTP from merascope.com and
cached under `cache/zcta/`. That avoids needing the 19 GB Hetzner data volume on
a dev box, and it guarantees we model exactly what the product actually serves.

**Labels** come from OSM (`man_made=data_centre`, `telecom=data_center`), re-queried
here rather than reused from `data/{STATE}/raw/datacenters.geojson`. The pipeline's
`01_basemap.py` requests `out center`, which throws the footprint polygon away. We
need `out geom`, because footprint area is the only thing in OSM that separates a
2.5 km2 Google campus from a single rack room in a downtown office tower — and
those two things are not the same siting decision.

In [ ]:
import sys, subprocess
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import dcmodel as D

D.CACHE.mkdir(exist_ok=True)
D.ZCTA_DIR.mkdir(parents=True, exist_ok=True)
print(len(D.STATES), "states")

## 1. Pull per-state ZCTA scores (skips anything already cached)

In [ ]:
import requests

for st in D.STATES:
    path = D.ZCTA_DIR / f"{st}.geojson"
    if path.exists() and path.stat().st_size > 0:
        continue
    url = f"https://merascope.com/data/{st}/zcta/grid_scores.geojson"
    r = requests.get(url, timeout=300); r.raise_for_status()
    path.write_bytes(r.content)
    print(f"{st}: {len(r.content):,} bytes")

total = sum(p.stat().st_size for p in D.ZCTA_DIR.glob("*.geojson"))
print(f"cached {len(list(D.ZCTA_DIR.glob('*.geojson')))} states, {total/1e6:.0f} MB")

## 2. Pull OSM data centers *with footprint geometry*

`cache/fetch_osm_dc.py` runs the national Overpass query. Overpass will often
504 on the first attempt at this size; the script retries.

In [ ]:
if not (D.CACHE / "osm_datacenters.geojson").exists():
    subprocess.run([sys.executable, "cache/fetch_osm_dc.py"], check=True)
    subprocess.run([sys.executable, "cache/build_dc_table.py"], check=True)

dc = D.load_datacenters()
print(f"{len(dc)} data-center features")
print(dc["osm_type"].value_counts().to_string())

### Footprint size distribution

This spans five orders of magnitude, which is the whole point. Note how few
features are genuinely hyperscale.

In [ ]:
a = dc["area_m2"].dropna()
print(f"polygons with area: {len(a)}   nodes (no footprint): {dc['area_m2'].isna().sum()}")
for q in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  p{q:<3} {np.percentile(a, q):>12,.0f} m2")
print(f"  max  {a.max():>12,.0f} m2")
print()
print(f">= 1 ha  (10,000 m2):  {(a >= 1e4).sum():>4}")
print(f">= 10 ha (100,000 m2): {(a >= 1e5).sum():>4}")
print()
print(dc.nlargest(10, "area_m2")[["name", "operator", "area_m2"]].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.log10(a), bins=50, color="#4c72b0", edgecolor="white", linewidth=0.4)
for t, lab in [(1e4, "1 ha"), (1e5, "10 ha")]:
    ax.axvline(np.log10(t), color="#c44e52", linestyle="--", linewidth=1.2)
    ax.text(np.log10(t), ax.get_ylim()[1]*0.92, f" {lab}", color="#c44e52", fontsize=9)
ax.set_xlabel("log10 footprint area (m2)"); ax.set_ylabel("OSM data-center features")
ax.set_title("Mapped US data centers span five orders of magnitude of footprint")
plt.tight_layout(); plt.show()

## 3. Load ZCTAs and attach labels

Two things to watch:

- **Cross-state duplicates.** `build_zcta.py` clips with `intersects`, not `within`,
  so a ZCTA straddling a state line lands in both states' files with *different*
  scores (normalization is state-relative). We keep the copy with the larger
  clipped area, i.e. the state the ZCTA mostly lies in.
- **Points, not polygons, for the join.** A large campus can overlap several ZCTAs;
  we attribute it to the one containing its representative point.

In [ ]:
g  = D.load_zctas()
df = D.attach_labels(g, dc)

df.drop(columns="geometry").to_csv(D.CACHE / "model_table.csv", index=False)
df[["zcta", "state", "geometry"]].to_file(D.CACHE / "zcta_geom.geojson", driver="GeoJSON")

for c in ["has_dc_any", "has_dc_10000", "has_dc_100000"]:
    print(f"{c:<16} positives={df[c].sum():>5}   base rate={100*df[c].mean():.2f}%")

Three candidate labels, and the choice matters:

| label | threshold | positives | base rate |
|---|---|---|---|
| `has_dc_any` | any mapped DC | 523 | 1.57% |
| `has_dc_10000` | >= 1 ha footprint | 204 | 0.61% |
| `has_dc_100000` | >= 10 ha footprint | 29 | 0.09% |

`has_dc_100000` is the label that matches what Merascope's customers actually
worry about — hyperscale campuses — but 29 positives is too few to fit and
validate honestly. `has_dc_10000` is the workable compromise; `has_dc_any`
is the most statistically comfortable and the least meaningful.